In [3]:
import os, glob, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
torch.manual_seed(42)

print(glob.glob("data/*.csv*"))                       # shows which label files exist

sp = pd.read_csv("data/splits.csv")
sp = sp[sp.label == 1].reset_index(drop=True)          # only patients that have pneumonia boxes
lab = pd.read_csv(glob.glob("data/stage_2_train_labels.csv*")[0])
lab = lab[lab.Target == 1]
boxes = {p: g[["x", "y", "width", "height"]].values for p, g in lab.groupby("patientId")}
S = 224 / 1024

def make_mask(pid):
    m = np.zeros((224, 224), np.float32)
    for x, y, w, h in boxes.get(pid, []):
        m[int(y * S):int(np.ceil((y + h) * S)), int(x * S):int(np.ceil((x + w) * S))] = 1
    return m

class Seg(Dataset):
    def __init__(self, d, train): self.d, self.train = d.reset_index(drop=True), train
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        pid = self.d.patientId[i]
        x = np.asarray(Image.open(f"data/processed/{pid}.png").convert("L"), np.float32) / 255
        y = make_mask(pid)
        if self.train and np.random.rand() < 0.5:
            x, y = x[:, ::-1].copy(), y[:, ::-1].copy()
        return torch.tensor(x)[None], torch.tensor(y)[None]

mk = lambda s, tr: DataLoader(Seg(sp[sp.split == s], tr), batch_size=8, shuffle=tr)
tr_dl, va_dl, te_dl = mk("train", True), mk("val", False), mk("test", False)
print({s: int((sp.split == s).sum()) for s in ["train", "val", "test"]})

['data\\splits.csv', 'data\\splits.csv.dvc', 'data\\stage_2_train_labels.csv.zip', 'data\\subset_ids.csv']
{'train': 210, 'val': 30, 'test': 60}


In [4]:
def blk(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                         nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU())

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1, self.e2, self.e3, self.b = blk(1, 16), blk(16, 32), blk(32, 64), blk(64, 128)
        self.u3, self.d3 = nn.ConvTranspose2d(128, 64, 2, 2), blk(128, 64)
        self.u2, self.d2 = nn.ConvTranspose2d(64, 32, 2, 2), blk(64, 32)
        self.u1, self.d1 = nn.ConvTranspose2d(32, 16, 2, 2), blk(32, 16)
        self.out = nn.Conv2d(16, 1, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(F.max_pool2d(e1, 2)); e3 = self.e3(F.max_pool2d(e2, 2))
        b = self.b(F.max_pool2d(e3, 2))
        d3 = self.d3(torch.cat([self.u3(b), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)

def dice(logit, y, eps=1e-6):
    p = torch.sigmoid(logit)
    return (2 * (p * y).sum((1, 2, 3)) + eps) / (p.sum((1, 2, 3)) + y.sum((1, 2, 3)) + eps)

net = UNet()
opt = torch.optim.Adam(net.parameters(), lr=1e-3)

def val_dice(dl):
    net.eval(); ds = []
    with torch.no_grad():
        for x, y in dl:
            p = (torch.sigmoid(net(x)) > 0.5).float()
            ds += ((2 * (p * y).sum((1, 2, 3)) + 1e-6) / (p.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1e-6)).tolist()
    return float(np.mean(ds))

In [5]:
os.makedirs("models", exist_ok=True)
best = -1
for ep in range(1, 16):
    net.train(); tot = 0
    for x, y in tr_dl:
        opt.zero_grad()
        lo = net(x)
        loss = F.binary_cross_entropy_with_logits(lo, y) + (1 - dice(lo, y)).mean()
        loss.backward(); opt.step(); tot += loss.item()
    v = val_dice(va_dl)
    print(f"epoch {ep}  loss {tot/len(tr_dl):.3f}  val Dice {v:.3f}")
    if v > best: best = v; torch.save(net.state_dict(), "models/unet_seg.pt")

epoch 1  loss 1.342  val Dice 0.211
epoch 2  loss 1.213  val Dice 0.279
epoch 3  loss 1.126  val Dice 0.141
epoch 4  loss 1.061  val Dice 0.421
epoch 5  loss 1.000  val Dice 0.455
epoch 6  loss 0.970  val Dice 0.325
epoch 7  loss 0.917  val Dice 0.298
epoch 8  loss 0.901  val Dice 0.435
epoch 9  loss 0.864  val Dice 0.401
epoch 10  loss 0.863  val Dice 0.444
epoch 11  loss 0.850  val Dice 0.423
epoch 12  loss 0.845  val Dice 0.465
epoch 13  loss 0.810  val Dice 0.419
epoch 14  loss 0.809  val Dice 0.471
epoch 15  loss 0.794  val Dice 0.436


In [ ]:
net.load_state_dict(torch.load("models/unet_seg.pt"))
print("test Dice:", round(val_dice(te_dl), 3), "on", len(te_dl.dataset), "pneumonia images")

net.eval(); x, y = next(iter(te_dl))
with torch.no_grad(): p = (torch.sigmoid(net(x)) > 0.5).float()
fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    for r, (m, t) in enumerate([(y[i, 0], "box mask (label)"), (p[i, 0], "model prediction")]):
        ax[r, i].imshow(x[i, 0], cmap="gray"); ax[r, i].imshow(m, alpha=0.4, cmap="Reds"); ax[r, i].set_title(t); ax[r, i].axis("off")
os.makedirs("docs", exist_ok=True)
plt.tight_layout(); plt.savefig("docs/segmentation_examples.png", dpi=110); plt.show()